In [907]:
import numpy as np
import pandas as pd
import plotly.express as px

In [908]:
np.random.seed(0)

In [909]:
threshold_min, threshold_max, threshold_delta = 0., 1., 0.1

In [910]:
def bayesian_update(priors):
    if np.sum(priors) == 0:
        return np.zeros_like(priors)
    return priors / np.sum(priors)

In [911]:
def manipulation_thresholds(thresholds, priors, c):
    if np.sum(priors) == 0:
        return thresholds
    thresholds_m = np.maximum(0, thresholds - (bayesian_update(priors) / c))

    if len(thresholds_m) > 1:
        priors_updated = []
        thresholds_updated = []
        thresholds_m_updated = []
        prev = -np.inf
        for i, threshold_m in enumerate(thresholds_m):
            if threshold_m <= prev:
                del priors_updated[i-1]
                del thresholds_updated[i-1]
                del thresholds_m_updated[i-1]

                priors_updated.append(priors[i-1] + priors[i])
            else:
                priors_updated.append(priors[i])
            thresholds_updated.append(thresholds[i])
            thresholds_m_updated.append(thresholds_m[i])
            prev = threshold_m
    else:
        priors_updated = priors
        thresholds_updated = thresholds
        thresholds_m_updated = thresholds_m

    return thresholds_m_updated, priors_updated, thresholds_updated

In [912]:
def balance_priors(priors, random=False):
    total = np.sum(priors)
    if total == 1:
        return priors
    indices = priors == 0
    remainder = 1 - total
    if random:
        p = np.random.rand(indices.sum())
        p = (p / p.sum()) * remainder
    else:
        p = remainder / indices.sum()
    priors[indices] = p
    return priors

In [913]:
def accuracy_loss(thresholds, x_manipulation, threshold_true):
    losses = []
    for i in range(len(thresholds)):
        if thresholds[i] < threshold_true:
            loss = (threshold_true - x_manipulation[i]) / (threshold_max - threshold_min)
        else:
            loss = np.abs(x_manipulation[i] - threshold_true) / (threshold_max - threshold_min)
        losses.append(loss)
    return np.array(losses)

In [914]:
thresholds = np.arange(threshold_min+threshold_delta, threshold_max, threshold_delta).round(4)

priors = np.zeros_like(thresholds)
priors[1] = 0.78
priors[7] = 0.15

balance_priors(priors, random=True)
print(np.sum(priors))
# assert np.sum(priors) == 1

pd.DataFrame({"threshold": thresholds, "priors": priors}).round(3).T

1.0


,0,1,2,3,4,5,6,7,8
threshold,0.10,0.20,0.300,0.400,0.50,0.600,0.700,0.80,0.900
priors,0.01,0.78,0.013,0.011,0.01,0.008,0.012,0.15,0.008


In [927]:
priors

array([0.00980328, 0.78      , 0.0127752 , 0.01076697, 0.00973307,
       0.00756761, 0.0115374 , 0.15      , 0.00781648])

In [931]:
c = 5
threshold_true = 0.5
n = len(thresholds)
# partitions = [[i] for i in range(n)]
partitions = [[0,1], [2,3], [4,5], [6,7,8]]

In [932]:
results = {
    "threshold": [],
    "prior": [],
    "partition": [],
    "accuracy_loss": [],
    "partition_loss": [],
    "manip_threshold": [],
}

for i, partition in enumerate(partitions):
    threshold_p = thresholds[partition]
    priors_p = priors[partition]
    x_manipulation_p, priors_p, threshold_p = manipulation_thresholds(threshold_p, priors_p, c)
    acc_loss = accuracy_loss(threshold_p, x_manipulation_p, threshold_true)
    partition_loss = np.dot(acc_loss, bayesian_update(priors_p))
    
    for ti, t in enumerate(threshold_p):
        results["threshold"].append(t)
        results["prior"].append(priors_p[ti])
        results["partition"].append(f"{i}")
        results["accuracy_loss"].append(acc_loss[ti])
        results["partition_loss"].append(partition_loss)
        results["manip_threshold"].append(x_manipulation_p[ti])

In [933]:
pd.DataFrame(results).T

,0,1,2,3,4,5,6
threshold,0.2,0.3,0.4,0.5,0.6,0.8,0.9
prior,0.789803,0.012775,0.010767,0.009733,0.007568,0.161537,0.007816
partition,0,1,1,2,2,3,3
accuracy_loss,0.497518,0.30853,0.19147,0.112517,0.012517,0.122856,0.390769
partition_loss,0.497518,0.254993,0.254993,0.068775,0.068775,0.135222,0.135222
manip_threshold,0.002482,0.19147,0.30853,0.387483,0.512517,0.622856,0.890769


In [926]:
print(f"True threshold: {threshold_true}")
fig = px.scatter(results, x = "threshold", y="partition_loss", color="partition", hover_data=["accuracy_loss"])

fig.show()

True threshold: 0.5


In [574]:
results

{'threshold': [],
 'partition': [],
 'accuracy_loss': [],
 'partition_loss': [],
 'manip_threshold': [],
 'partition_type': []}

In [511]:
threshold_partitions = []
while True:
    best_loss = np.inf
    best_pair = None

    for i in range(len(partitions)):
        p = partitions[i]
        threshold_p = thresholds[p]
        priors_p = priors[p]
        x_manipulation_p = manipulation_thresholds(threshold_p, priors_p, c)
        acc_loss_p = accuracy_loss(threshold_p, x_manipulation_p, threshold_true).item()
        for j in range(i+1, len(partitions)):
            q = partitions[j]
            threshold_q = thresholds[q]
            priors_q = priors[q]
            x_manipulation_q = manipulation_thresholds(threshold_q, priors_q, c)
            acc_loss_q = accuracy_loss(threshold_q, x_manipulation_q, threshold_true).item()

            pq = [p[0],q[0]]
            threshold_pq = thresholds[pq]
            priors_pq = priors[pq]
            x_manipulation_pq = manipulation_thresholds(threshold_pq, priors_pq, c)
            acc_loss_pq = accuracy_loss(threshold_pq, x_manipulation_pq, threshold_true)
            acc_loss_merged = np.dot(acc_loss_pq, bayesian_update(priors_pq))
            
            if acc_loss_merged < min(acc_loss_p, acc_loss_q) and acc_loss_merged < best_loss:
                best_pair = [i, j]
                best_loss = acc_loss_merged
    
    if best_pair is None:
        break
    i, j = best_pair
    threshold_partitions.append(best_pair)

    del partitions[i]
    del partitions[i]

threshold_partitions.extend(partitions)

In [512]:
threshold_partitions

[[3, 4], [0, 1], [2], [5], [6], [7], [8]]

In [513]:
for i, partition in enumerate(threshold_partitions):
    threshold_p = thresholds[partition]
    priors_p = priors[partition]
    x_manipulation_p = manipulation_thresholds(threshold_p, priors_p, c)
    acc_loss = accuracy_loss(threshold_p, x_manipulation_p, threshold_true)
    partition_loss = np.dot(acc_loss, bayesian_update(priors_p))
    
    for ti, t in enumerate(threshold_p):
        results["threshold"].append(t)
        results["partition"].append(f"{i}")
        results["accuracy_loss"].append(acc_loss[ti])
        results["partition_loss"].append(partition_loss)
        results["manip_threshold"].append(x_manipulation_p[ti])
        results["partition_type"].append("merged")

In [925]:
print(f"True threshold: {threshold_true}")
fig = px.scatter(results, x = "threshold", y="partition_loss", color="partition", symbol="partition_type", hover_data=["accuracy_loss"])

fig.show()

True threshold: 0.5


ValueError: Value of 'symbol' is not the name of a column in 'data_frame'. Expected one of ['threshold', 'prior', 'partition', 'accuracy_loss', 'partition_loss', 'manip_threshold'] but received: partition_type

In [ ]:
size = input()
if size.isdigit():
    size = int(size)

False